# Project 2 — Supply Chain Causal Inference


Project 1 answered "how many units will sell?" This project answers a different, equally critical question: "what is the causal effect of an intervention on demand?" — 

 The dataset encodes SNAP (US Supplemental Nutrition Assistance Program) days via `snap_CA`, `snap_TX`, `snap_WI`. On SNAP days, food-assistance benefits can be spent. This is a real-world demand intervention with three properties that make it ideal for causal analysis:
1. It is scheduled by policy (the first ~10 days of each month), not by Walmart's marketing, reducing the usual price/promotion confounding.
2. It plausibly affects FOODS but not HOBBIES giving us a built-in control group (a placebo check).
3. SNAP calendars differ across states, enabling difference-in-differences.

If SNAP days cause a real, repeatable food-demand lift, that lift must be pre-positioned as inventory and staffing — not chased reactively. Sizing that causal effect (not just correlating it) is what prevents both stockouts on high-demand days and overstock on normal ones.

In [1]:
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13,4); plt.rcParams["axes.grid"] = True

cal = pd.read_csv("calendar.csv")
sales = pd.read_csv("sales_train_evaluation.csv")
day_cols = [c for c in sales.columns if c.startswith("d_")]
id_cols  = ["id","item_id","dept_id","cat_id","store_id","state_id"]
print("series:", len(sales), "| days:", len(day_cols))

series: 30490 | days: 1941


## 1. Inference vs. prediction 

A predictive model can score SNAP as 'important' simply because SNAP days correlate with paydays, weekends, or month-start shopping. Correlation is not the causal effect. Causal inference isolates the portion of the demand change attributable to SNAP, holding confounders fixed. We build a state-level daily FOODS panel and tag each day with its state's SNAP flag, day-of-week, and event context (the confounders).

In [2]:
# Build state x day FOODS and HOBBIES daily totals (cheap: column sums on the wide matrix)
def cat_state_panel(cat):
    sub = sales[sales.cat_id == cat]
    g = sub.groupby("state_id", observed=True)[day_cols].sum().T   # days x states
    g.index = day_cols
    long = g.reset_index().melt(id_vars="index", var_name="state_id", value_name="sales").rename(columns={"index":"d"})
    long["cat"] = cat
    return long

panel = pd.concat([cat_state_panel("FOODS"), cat_state_panel("HOBBIES")], ignore_index=True)
cal_keep = cal[["d","date","wday","month","year","event_name_1","event_type_1",
                "snap_CA","snap_TX","snap_WI"]].copy()
panel = panel.merge(cal_keep, on="d", how="left")
panel["date"] = pd.to_datetime(panel["date"])
# Map each row to ITS state's SNAP flag
panel["snap"] = panel.apply(lambda r: r[f"snap_{r.state_id}"], axis=1).astype(int)
panel["dow"] = panel["date"].dt.dayofweek
panel["has_event"] = (panel["event_name_1"].notna()).astype(int)
panel = panel[["date","state_id","cat","sales","snap","dow","month","year","has_event"]]
print(panel.shape); panel.head()

(11646, 9)


,date,state_id,cat,sales,snap,dow,month,year,has_event
0,2011-01-29,CA,FOODS,10101,0,5,1,2011,0
1,2011-01-30,CA,FOODS,9862,0,6,1,2011,0
2,2011-01-31,CA,FOODS,6944,0,0,1,2011,0
3,2011-02-01,CA,FOODS,7864,1,1,2,2011,0
4,2011-02-02,CA,FOODS,7178,1,2,2,2011,0


## 2. Naive estimate 

First, the tempting-but-wrong approach: just compare average FOODS sales on SNAP vs non-SNAP days. This is the observational difference, and it is confounded.

In [3]:
foods = panel[panel.cat == "FOODS"]
naive_on  = foods.loc[foods.snap==1,"sales"].mean()
naive_off = foods.loc[foods.snap==0,"sales"].mean()
print("Naive FOODS: SNAP=%.0f  non-SNAP=%.0f  =>  +%.1f%% lift (BIASED)"
      % (naive_on, naive_off, 100*(naive_on/naive_off-1)))

# Confounder check: are SNAP days distributed differently across the week / month?
print("\nSNAP-day share by day-of-week (0=Mon):")
print(foods.groupby("dow")["snap"].mean().round(2).to_string())
print("\nSNAP-day share by month-half:")
foods2 = foods.assign(half=np.where(foods.date.dt.day<=15,"1st half","2nd half"))
print(foods2.groupby("half")["snap"].mean().round(2).to_string())

Naive FOODS: SNAP=8749  non-SNAP=7462  =>  +17.3% lift (BIASED)

SNAP-day share by day-of-week (0=Mon):
dow
0    0.32
1    0.34
2    0.33
3    0.33
4    0.33
5    0.33
6    0.33

SNAP-day share by month-half:
half
1st half    0.67
2nd half    0.00


**Why the naive number is biased:** SNAP days cluster in the first half of the month** and interact with day-of-week shopping rhythms. Those same factors independently drive grocery demand (paychecks, weekend stock-ups). So the raw +X% mixes the true SNAP effect with calendar confounding. We need designs that break that confounding: (A) a placebo/control category, (B) difference-in-differences, (C) covariate adjustment via regression and propensity weighting.

## 3. Placebo test with a control category (HOBBIES)

SNAP benefits buy food, not toys. If our method is sound, it should find a large effect on FOODS and a near-zero effect on HOBBIES. A non-zero HOBBIES 'effect' would reveal residual confounding.

In [4]:
for c in ["FOODS","HOBBIES"]:
    g = panel[panel.cat==c]
    on, off = g.loc[g.snap==1,"sales"].mean(), g.loc[g.snap==0,"sales"].mean()
    print("%-8s SNAP lift: %+.1f%%" % (c, 100*(on/off-1)))

FOODS    SNAP lift: +17.3%
HOBBIES  SNAP lift: +2.5%


**Placebo read-out:** FOODS shows a clear positive lift while HOBBIES shows little to none strong evidence the effect is food-specific and not a generic 'SNAP days are just busy days' artifact. This control-group logic is the backbone of the difference-in-differences estimator next.

## 4. Difference-in-Differences 

DiD removes time-invariant differences and common shocks by comparing the treated group's change to a control group's change. Here we treat FOODS as 'treated by SNAP' and HOBBIES as control, comparing SNAP vs non-SNAP days. The interaction term `snap × is_food` is the causal estimate.

In [5]:
import statsmodels.formula.api as smf
did_df = panel.copy()
did_df["is_food"] = (did_df.cat=="FOODS").astype(int)
# log sales stabilizes variance & gives a % interpretation
did_df["lsales"] = np.log(did_df["sales"].clip(lower=1))
did = smf.ols("lsales ~ snap * is_food + C(dow) + C(month) + has_event", data=did_df).fit()
print(did.summary().tables[1])
beta = did.params["snap:is_food"]
print("\nDiD causal SNAP effect on FOODS: %+.1f%% (95%% CI: %+.1f%% .. %+.1f%%)"
      % (100*beta, 100*did.conf_int().loc["snap:is_food",0], 100*did.conf_int().loc["snap:is_food",1]))

                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          6.8663      0.020    339.157      0.000       6.827       6.906
C(dow)[T.1]       -0.1038      0.017     -6.195      0.000      -0.137      -0.071
C(dow)[T.2]       -0.0969      0.017     -5.787      0.000      -0.130      -0.064
C(dow)[T.3]       -0.0898      0.017     -5.370      0.000      -0.123      -0.057
C(dow)[T.4]        0.0236      0.017      1.405      0.160      -0.009       0.057
C(dow)[T.5]        0.2274      0.017     13.558      0.000       0.195       0.260
C(dow)[T.6]        0.1775      0.017     10.623      0.000       0.145       0.210
C(month)[T.2]      0.0445      0.022      2.045      0.041       0.002       0.087
C(month)[T.3]      0.0136      0.021      0.639      0.523      -0.028       0.055
C(month)[T.4]      0.0240      0.021      1.117      0.264      -0.018       0.066
C(mo

**DiD interpretation:** The `snap:is_food` coefficient is the SNAP effect on FOODS net of anything that moves both categories (day-of-week, seasonality, events) and net of fixed category differences. Because we control for `C(dow)`, `C(month)` and `has_event`, this estimate strips out the calendar confounding that inflated the naive number. The coefficient on `snap` alone (the HOBBIES/control response) should be small — the DiD's identifying assumption is parallel trends: absent SNAP, FOODS and HOBBIES would have moved together.

## 5. Covariate adjustment + Propensity-Score Weighting 

When we only have observational data (no randomized experiment), we estimate the propensity of a day being a SNAP day from its covariates, then reweight so SNAP and non-SNAP days have comparable covariate distributions (inverse-propensity weighting, IPW). This emulates a randomized comparison.

In [6]:
from sklearn.linear_model import LogisticRegression
fd = foods.copy()
X = pd.get_dummies(fd[["dow","month","has_event"]], columns=["dow","month"], drop_first=True).astype(float)
t = fd["snap"].values
ps = LogisticRegression(max_iter=1000).fit(X, t).predict_proba(X)[:,1]
ps = np.clip(ps, 0.05, 0.95)                      # trim extreme weights for stability

w = np.where(t==1, 1/ps, 1/(1-ps))                # IPW (ATE) weights
y = fd["sales"].values.astype("float64")
gap = y[t==1].mean() - y[t==0].mean()
att_ipw = np.average(y[t==1], weights=w[t==1]) - np.average(y[t==0], weights=w[t==0])
print("Naive difference (units/day):          %+.0f" % gap)
print("IPW-adjusted causal effect (ATE):      %+.0f units/day" % att_ipw)
print("change vs naive after covariate balance: %+.1f%%" % (100*(att_ipw/gap - 1)))

Naive difference (units/day):          +1287
IPW-adjusted causal effect (ATE):      +1288 units/day
change vs naive after covariate balance: +0.0%


In [7]:
# Covariate balance: weighting should shrink standardized mean differences toward 0
def smd(col):
    a,b = X[col].values[t==1], X[col].values[t==0]
    raw = (a.mean()-b.mean())/np.sqrt((a.var()+b.var())/2 + 1e-9)
    wa = np.average(X[col].values[t==1], weights=w[t==1])
    wb = np.average(X[col].values[t==0], weights=w[t==0])
    adj = (wa-wb)/np.sqrt((a.var()+b.var())/2 + 1e-9)
    return raw, adj
bal = pd.DataFrame({c: smd(c) for c in X.columns[:8]}, index=["raw_SMD","weighted_SMD"]).T
print(bal.round(3).to_string())

           raw_SMD  weighted_SMD
has_event    0.025         0.001
dow_1        0.013         0.000
dow_2       -0.004         0.000
dow_3       -0.002        -0.000
dow_4       -0.002         0.000
dow_5       -0.002         0.000
dow_6        0.007        -0.001
month_2      0.032         0.002


**Propensity-weighting takeaway.** Re-weighting on day-of-week / month / event covariates barely moves the FOODS estimate (IPW ~naive). That is itself an informative result: SNAP timing is approximately as-if-random with respect to these calendar features, the program runs on a fixed monthly schedule, not in response to weekday/seasonal demand, so within the FOODS series there is little measured confounding to remove. The balance table confirms covariates were already fairly balanced and the standardized mean differences stay small after weighting. We trimmed propensities to [0.05, 0.95] so near-deterministic days don't inflate variance, a standard robustness step. The real adjustment comes from DiD (17.3% → 15.1%), which strips the common cross-category component that single-series weighting cannot. The agreement in direction and magnitude across three designs with different assumptions, placebo control, DiD, and IPW  is what makes the causal claim credible. 

## 6. Designing a  A/B test:

Observational methods correct for known confounders; a randomized experiment neutralizes all confounders by design. Here is how I would run a real experiment.

In [8]:
# Power analysis: how many store-days do we need to detect a 3% demand lift?
from statsmodels.stats.power import TTestIndPower
daily_store = (panel[panel.cat=="FOODS"].groupby(["state_id","date"])["sales"].sum())
mu, sd = daily_store.mean(), daily_store.std()
mde = 0.03 * mu                          # minimum detectable effect = 3% of mean daily demand
effect_size = mde / sd
n = TTestIndPower().solve_power(effect_size=effect_size, alpha=0.05, power=0.80, alternative="two-sided")
print("baseline mean daily FOODS/store: %.0f (sd %.0f)" % (mu, sd))
print("target MDE: %.0f units (3%%) | standardized effect: %.3f" % (mde, effect_size))
print("required store-days PER ARM for 80%% power at alpha=0.05: %.0f" % np.ceil(n))

baseline mean daily FOODS/store: 7886 (sd 2400)
target MDE: 237 units (3%) | standardized effect: 0.099
required store-days PER ARM for 80% power at alpha=0.05: 1616


---
## Project 2 conclusion
Using SNAP days as a natural experiment, we showed why the **naive** SNAP-vs-non-SNAP comparison is **confounded**, then triangulated the true causal effect with three complementary tools: a placebo control category (HOBBIES ~ no effect), difference-in-differences (FOODS effect net of calendar shocks), and propensity-score / IPW adjustment (which quantified how much of the raw gap was confounding). Finally we laid out a rigorous prospective A/B test, with power analysis, for a supply-chain replenishment intervention.

**Takeaway:** 
Project 1 demonstrates the predictive ML lifecycle (engineering -> EDA ->  time-series modeling ->  leakage-free validation ->  inventory-aware metrics); 
Project 2 demonstrates the causal/experimentation lifecycle (confounding ->  DiD ->  propensity methods ->  experiment design). Together they cover the two questions : "what will demand be" and "what should we do about it"